# 🎓 AI Career & Placement System – Jupyter Walkthrough
**By Ankit** · ankitchadrabad9389@gmail.com

Run the cells top to bottom (**Shift+Enter**). This notebook uses the same `src/` modules as the Streamlit app, so you can see every step of the pipeline.
> ⚠️ The ML model uses **synthetic** data and company criteria are **illustrative**.

## 1. Setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))          # project root
os.environ["SQLITE_PATH"] = os.path.abspath("demo_notebook.db")   # separate demo DB
import pandas as pd
import plotly.express as px
from src import db, ml_model as ml, skills_extractor as sx, career, readiness as rd, companies as co, roadmap as rm, interview as iv
db.init_db()
print("Database ready (SQLite). Companies loaded:", len(db.query("SELECT * FROM companies")))

## 2. Synthetic dataset (EDA)

In [ ]:
df = ml.load_data()
print(df.shape, "| placement rate:", round(df.placed.mean()*100, 1), "%")
df.head()

In [ ]:
df["cgpa_band"] = pd.cut(df.cgpa, [0, 6, 7, 8, 9, 10]).astype(str)
px.bar(df.groupby("cgpa_band").placed.mean().mul(100).reset_index(), x="cgpa_band", y="placed",
       title="Placement rate % by CGPA band (synthetic data)").show()

## 3. Train & compare ML models

In [ ]:
bundle = ml.train()
print("Best model:", bundle["best_model"])
print("CV ROC-AUC:", {k: round(v, 3) for k, v in bundle["cv_auc"].items()})
pd.Series(bundle["metrics"]).drop(["confusion"]).round(3)

In [ ]:
imp = pd.Series(bundle["importances"]).sort_values()
px.bar(imp, orientation="h", title="Feature importance").show()

## 4. Register a student & build the profile

In [ ]:
db.run("DELETE FROM students WHERE email='ankit.demo@example.com'")
sid = db.create_student("Ankit", "ankit.demo@example.com", "secret123")
db.update_profile(sid, cgpa=8.1, tenth_pct=86, twelfth_pct=82, backlogs=0, internships=1, projects=3,
                  certifications=2, hackathons=1, coding_score=62, aptitude=60, communication=7, target_role="Data Scientist")
S = db.get_student(sid)
S

## 5. Resume skill extraction

In [ ]:
resume_text = """Ankit | ankit.demo@example.com | CGPA: 8.1
Skills: Python, Pandas, NumPy, SQL, MySQL, Scikit-learn, Machine Learning, Git, GitHub, Excel
Projects: House price prediction (machine learning), Sales dashboard
Internship at a startup (data analysis)"""
# To use a real file instead:  text = sx.extract_text(open("resume.pdf","rb").read(), "resume.pdf")
skills = sx.extract_skills(resume_text)
print("Skills:", skills); print("Meta:", sx.extract_meta(resume_text))
db.set_skills(sid, skills, "resume")

## 6. Readiness score + ML placement probability + what-if

In [ ]:
skills = db.get_skills(sid)
P = {f: float(S[f]) for f in ml.FEATURES if f != "num_skills"}
role = S["target_role"]
ready, parts, band = rd.compute(P, skills, role)
prob = ml.predict_proba(P, len(skills))
print(f"Readiness: {ready}/100 ({band}) | ML placement probability: {prob*100:.1f}%")
print(parts)
ml.what_if(P, len(skills))

## 7. Career roles & company compatibility

In [ ]:
rec = career.recommend_roles(skills)
display(rec[["role", "match_pct", "missing"]])
px.bar(rec, x="match_pct", y="role", orientation="h", title="Role compatibility %").show()

In [ ]:
comp = co.check(P, skills, ready)
comp[["Company", "Tier", "Package_LPA", "Eligible", "Compatibility", "Missing_Skills"]]

## 8. Skill-gap roadmap & interview questions

In [ ]:
road, weeks = rm.build_roadmap(skills, role, hours_per_week=10)
print("Total weeks:", weeks)
road

In [ ]:
tech, hr = iv.questions(role, skills, per_skill=1)
for s, q, h in tech[:6]: print(f"[{s}] {q}  (hint: {h})")

## 9. Save the PDF report & save prediction history

In [ ]:
from src import report
db.save_prediction(sid, prob, ready, role)
pdf = report.build_pdf(S, role, ready, parts, band, prob, rec, comp, road, weeks)
open("student_report.pdf", "wb").write(pdf)
print("Saved notebooks/student_report.pdf"); db.prediction_history(sid)

## 10. Run the full web app
Streamlit runs best from a **terminal**, not from inside a notebook:
```
cd ..            # project root
streamlit run app.py
```